#  RFM Calculation
## Customer Retention & RFM Analysis

#### This notebook calculates RFM (Recency, Frequency, Monetary) values
#### for every customer in the dataset.
#### Business purpose: Classify all customers by how recently they bought,
#### how often they buy, and how much they spend — so we can identify
#### which customers need retention attention and which should be rewarded.



In [74]:
import pandas as pd
import numpy as np 

In [75]:
df = pd.read_csv("customer_retail_clean.csv")

In [76]:
df['invoicedate'] = pd.to_datetime(df['invoicedate'])

In [77]:
df

,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country,revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.40
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.80
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.00
...,...,...,...,...,...,...,...,...,...
779420,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680,France,12.60
779421,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680,France,16.60
779422,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680,France,16.60
779423,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680,France,14.85


## 1. Data Validation

In [78]:
# Quick sanity check — confirm it matches what you cleaned
print(f"Rows:              {len(df):,}")
print(f"Unique customers:  {df['customer_id'].nunique():,}")
print(f"Date range:        {df['invoicedate'].min().date()} to {df['invoicedate'].max().date()}")
print(f"Total revenue:     £{df['revenue'].sum():,.2f}")

Rows:              779,425
Unique customers:  5,878
Date range:        2009-12-01 to 2011-12-09
Total revenue:     £17,374,804.27


In [79]:
print(df.shape)
print(df.dtypes)
print(df.isnull().sum())
df.head()

(779425, 9)
invoice                 int64
stockcode              object
description            object
quantity                int64
invoicedate    datetime64[ns]
price                 float64
customer_id             int64
country                object
revenue               float64
dtype: object
invoice        0
stockcode      0
description    0
quantity       0
invoicedate    0
price          0
customer_id    0
country        0
revenue        0
dtype: int64


,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country,revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


In [80]:
#  Observation:
# Dataset loads correctly with 779,425 rows across 5,878 unique customers.
# Date range confirms December 2009 to December 2011 — two full years of
# transaction history available for RFM and cohort analysis.

## 2. Reference Date

In [81]:
# Reference date — the anchor point for all Recency calculations
#
# Rule: always use the maximum date in the dataset plus one day.
# Never use today's date on historical data.
#
# Reason: if the dataset ends in December 2011 and we use today's date,
# every single customer would appear to have a Recency of 4,000+ days.
# Their scores would all collapse to 1, making Recency meaningless.
# Using the dataset's own maximum date preserves the relative differences
# between customers that RFM scoring depends on.

reference_date = df['invoicedate'].max() + pd.Timedelta(days=1)
print(f"Reference date: {reference_date.date()}")
print(f"This means Recency = 1 means they bought on {df['invoicedate'].max().date()}")

Reference date: 2011-12-10
This means Recency = 1 means they bought on 2011-12-09


In [82]:
#  Observation:  
# Reference date is set to December 10, 2011 — one day after the last
# transaction in the dataset. A customer who bought on December 9, 2011
# receives Recency = 1 day, the best possible value.
# All Recency calculations in this notebook are measured relative to this date.


## 3. RFM Base Calculation

In [83]:

# Collapse all transactions into one row per customer
# with their R, F, M base values

rfm_base = df.groupby('customer_id').agg(
    last_purchase_date  = ('invoicedate', 'max'),
    first_purchase_date = ('invoicedate', 'min'),
    Frequency           = ('invoice',     'nunique'),
    Monetary            = ('revenue',     'sum'),
    avg_order_value     = ('revenue',     'mean'),
    total_items         = ('quantity',    'sum'),
    unique_products     = ('stockcode',   'nunique')
).reset_index()

# Calculate Recency from reference date
rfm_base['Recency'] = (reference_date - rfm_base['last_purchase_date']).dt.days

# Calculate customer tenure — how long have they been a customer?
rfm_base['tenure_days'] = (
    rfm_base['last_purchase_date'] - rfm_base['first_purchase_date']
).dt.days

print(f"Unique customers in RFM table: {len(rfm_base):,}")
print()
print("RFM base statistics:")
print(rfm_base[['Recency', 'Frequency', 'Monetary']].describe().round(2))




Unique customers in RFM table: 5,878

RFM base statistics:
       Recency  Frequency   Monetary
count  5878.00    5878.00    5878.00
mean    201.33       6.29    2955.90
std     209.34      13.01   14440.85
min       1.00       1.00       2.95
25%      26.00       1.00     342.28
50%      96.00       3.00     867.74
75%     380.00       7.00    2248.30
max     739.00     398.00  580987.04


In [84]:

# Business Observation

# RFM metrics were calculated for 5,878 unique customers,
# creating a customer-level dataset for retention and segmentation analysis.

# The average customer has not made a purchase for approximately
# 201 days, indicating the presence of a sizable inactive customer population.

# Customer purchasing behavior varies significantly across the customer base.
# While the median customer placed only 3 orders, the most active customer
# placed 398 orders, highlighting substantial differences in customer loyalty.

# Revenue contribution is also highly uneven.
# The median customer generated approximately 868 in revenue,
# whereas the highest-value customer generated over 580K,
# suggesting strong revenue concentration among a small group of customers.

# Recency ranges from 1 day to 739 days,
# indicating that the dataset contains both recently active customers
# and customers who have likely churned.

# These findings confirm that customers differ significantly in terms of
# activity, purchase frequency, and spending behavior,
# making RFM segmentation essential for identifying Champions,
# Loyal Customers, At-Risk Customers, and Lost Customers.

## 4. Monetary Distribution & Outlier Assessment

In [85]:

# Check for outliers that could distort RFM scoring
print("Monetary distribution:")
print(f"  25th percentile: £{rfm_base['Monetary'].quantile(0.25):,.2f}")
print(f"  50th percentile: £{rfm_base['Monetary'].quantile(0.50):,.2f}")
print(f"  75th percentile: £{rfm_base['Monetary'].quantile(0.75):,.2f}")
print(f"  95th percentile: £{rfm_base['Monetary'].quantile(0.95):,.2f}")
print(f"  99th percentile: £{rfm_base['Monetary'].quantile(0.99):,.2f}")
print(f"  Maximum:         £{rfm_base['Monetary'].max():,.2f}")

# How many customers are above £10,000?
high_value = rfm_base[rfm_base['Monetary'] > 10000]
print(f"\nCustomers spending over £10,000: {len(high_value):,} ({len(high_value)/len(rfm_base)*100:.1f}%)")

Monetary distribution:
  25th percentile: £342.28
  50th percentile: £867.74
  75th percentile: £2,248.30
  95th percentile: £9,374.23
  99th percentile: £29,205.90
  Maximum:         £580,987.04

Customers spending over £10,000: 261 (4.4%)


In [86]:
# Business Observation

# Customer spending is highly skewed, with a small group of customers
# generating substantially more revenue than the majority of the customer base.

# The median customer generated approximately £868 in revenue,
# while the top 1% of customers generated more than £29,206 each.

# The highest-value customer generated over £580K in revenue,
# highlighting extreme revenue concentration among a small number
# of customers.

# Only 261 customers (4.4% of the customer base) spent more than £10,000,
# indicating that a relatively small segment contributes a significant
# share of total business revenue.

# These findings reinforce the importance of identifying and retaining
# high-value customers, as losing even a small number of them could
# have a substantial impact on overall revenue performance.

# The Monetary distribution contains several high-value spending customers,
# indicating the presence of revenue outliers.

# These outliers appear to represent genuine customer behavior rather than
# data quality issues, as they are valid transactions generated by
# real customers.

## 5. RFM Scoring (1–5 Scale)

In [87]:

# ── R_Score: Recency ──────────────────────────────────────────────────────
# INVERTED: lower Recency days = bought more recently = better = score 5
# A customer who bought yesterday gets R_Score = 5
# A customer who bought 300 days ago gets R_Score = 1
rfm_base['R_Score'] = pd.qcut(
    rfm_base['Recency'],
    q=5,
    labels=[5, 4, 3, 2, 1]   # 5 = most recent
)

# ── F_Score: Frequency ────────────────────────────────────────────────────
# Higher Frequency = better = score 5
# Using rank(method='first') to handle ties — customers with the same
# frequency get different ranks based on order they appear
# This prevents pd.qcut from failing on duplicate values
rfm_base['F_Score'] = pd.qcut(
    rfm_base['Frequency'].rank(method='first'),
    q=5,
    labels=[1, 2, 3, 4, 5]   # 5 = most frequent
)

# ── M_Score: Monetary ─────────────────────────────────────────────────────
# Higher Monetary = better = score 5
rfm_base['M_Score'] = pd.qcut(
    rfm_base['Monetary'],
    q=5,
    labels=[1, 2, 3, 4, 5]   # 5 = highest spender
)

# Convert to integer for easier comparison
rfm_base[['R_Score', 'F_Score', 'M_Score']] = (
    rfm_base[['R_Score', 'F_Score', 'M_Score']].astype(int)
)

# Create composite score
rfm_base['RFM_Total'] = rfm_base['R_Score'] + rfm_base['F_Score'] + rfm_base['M_Score']

print("Score distributions — should each show roughly equal counts:")
print("\nR_Score:")
print(rfm_base['R_Score'].value_counts().sort_index())
print("\nF_Score:")
print(rfm_base['F_Score'].value_counts().sort_index())
print("\nM_Score:")
print(rfm_base['M_Score'].value_counts().sort_index())

Score distributions — should each show roughly equal counts:

R_Score:
R_Score
1    1175
2    1172
3    1167
4    1176
5    1188
Name: count, dtype: int64

F_Score:
F_Score
1    1176
2    1175
3    1176
4    1175
5    1176
Name: count, dtype: int64

M_Score:
M_Score
1    1176
2    1175
3    1176
4    1175
5    1176
Name: count, dtype: int64


In [88]:
# Business Observation

# Customers were successfully divided into five approximately
# equal-sized groups for each RFM metric using quintile based scoring.

# The balanced score distributions confirm that the RFM model
# effectively separates customers across different levels of
# recency, purchase frequency, and spending behavior.

# Recency follows an inverted scoring approach, where the most
# recently active customers receive the highest score, while
# Frequency and Monetary assign higher scores to customers with
# greater purchase activity and spending.

# These standardized scores provide a solid foundation for
# customer segmentation and the identification of Champion,
# Loyal, At-Risk, and Lost customer groups.

## 6. Scoring Validation

In [89]:
# Sanity check: Champions (high R, F, M) should have low Recency days
# and high Frequency and Monetary values
print("Score 5-5-5 customers (potential Champions):")
top_customers = rfm_base[
    (rfm_base['R_Score'] == 5) &
    (rfm_base['F_Score'] == 5) &
    (rfm_base['M_Score'] == 5)
][['customer_id', 'Recency', 'Frequency', 'Monetary', 'RFM_Total']]
print(top_customers.sort_values('Monetary', ascending=False).head(10))

print(f"\nTotal 5-5-5 customers: {len(top_customers):,}")
print()

# Score 1-1-1 customers (worst on all dimensions)
print("Score 1-1-1 customers (potential Lost):")
bottom_customers = rfm_base[
    (rfm_base['R_Score'] == 1) &
    (rfm_base['F_Score'] == 1) &
    (rfm_base['M_Score'] == 1)
][['customer_id', 'Recency', 'Frequency', 'Monetary', 'RFM_Total']]
print(bottom_customers.sort_values('Recency', ascending=False).head(5))


Score 5-5-5 customers (potential Champions):
      customer_id  Recency  Frequency   Monetary  RFM_Total
5692        18102        1        145  580987.04         15
2277        14646        2        151  528602.52         15
1789        14156       10        156  313437.62         15
2538        14911        1        398  291420.81         15
5050        17450        8         51  244784.25         15
1331        13694        4        143  195640.69         15
5109        17511        3         60  172132.87         15
4295        16684        4         55  147142.77         15
2685        15061        4        127  126389.02         15
5540        17949        1        118  117314.08         15

Total 5-5-5 customers: 469

Score 1-1-1 customers (potential Lost):
      customer_id  Recency  Frequency  Monetary  RFM_Total
289         12636      739          1    141.00          3
3454        15833      738          1     80.40          3
2285        14654      738          1    246.86  

In [90]:
# Business Observation

# A validation check was performed to confirm that the RFM scoring
# correctly identifies both high-value and low-value customer groups.

# Customers with the highest possible RFM score (5-5-5)
# exhibit extremely recent purchasing behavior,
# high order frequency, and substantial revenue contribution.

# For example, Customer 18102 generated over £580K in revenue
# across 145 orders and made a purchase just one day before
# the dataset end date, making them a clear Champion customer.

# In total, 469 customers achieved the maximum RFM score,
# representing approximately 8% of the customer base.

# Conversely, customers with the lowest possible score (1-1-1)
# have not purchased for nearly two years,
# placed only a single order, and generated very little revenue.

# These results confirm that the RFM scoring model is functioning
# as expected and provides a reliable foundation for customer
# segmentation and retention analysis.

## At Risk Revenue Quantification 

In [91]:
# ── At-Risk Revenue Quantification ────────────────────────────────────────
# Business question: How much revenue is at risk from inactive customers?
# Define at-risk: R_Score of 1 or 2 (bottom 40% on Recency)
# AND previously had meaningful spend (M_Score 3 or above)

at_risk = rfm_base[
    (rfm_base['R_Score'] <= 2) &
    (rfm_base['M_Score'] >= 3)
].copy()

total_revenue      = rfm_base['Monetary'].sum()
at_risk_revenue    = at_risk['Monetary'].sum()
at_risk_pct        = at_risk_revenue / total_revenue * 100

print("=" * 50)
print("AT-RISK REVENUE SUMMARY")
print("=" * 50)
print(f"At-risk customers:          {len(at_risk):,}")
print(f"Their historical revenue:   £{at_risk_revenue:,.2f}")
print(f"As % of total revenue:      {at_risk_pct:.1f}%")
print(f"Average revenue per customer: £{at_risk['Monetary'].mean():,.2f}")

# Win-back projection
recovery_rate     = 0.30
campaign_discount = 0.15
recovered         = at_risk_revenue * recovery_rate
campaign_cost     = at_risk['Monetary'].mean() * campaign_discount * len(at_risk) * recovery_rate

print(f"\nIf 30% respond to win-back campaign:")
print(f"  Recoverable revenue:  £{recovered:,.2f}")
print(f"  Estimated ROI:        {recovered/campaign_cost:.0f}x")

AT-RISK REVENUE SUMMARY
At-risk customers:          859
Their historical revenue:   £1,836,190.44
As % of total revenue:      10.6%
Average revenue per customer: £2,137.59

If 30% respond to win-back campaign:
  Recoverable revenue:  £550,857.13
  Estimated ROI:        7x


In [92]:
# Business Observation

# A total of 859 customers were identified as At-Risk,
# representing customers who have not purchased recently
# but historically generated meaningful revenue.

# Collectively, these customers contributed approximately
# £1.84 million in historical revenue, accounting for
# 10.6% of total business revenue.

# On average, each At-Risk customer generated
# approximately £2,138, making them significantly more
# valuable than low-spending customers who have churned.

# This suggests that a relatively small group of inactive
# customers represents a substantial revenue opportunity
# for the business.

# Assuming a conservative 30% success rate from a targeted
# win-back campaign, approximately £551K of revenue could
# potentially be recovered.

# Based on the estimated campaign cost, the projected return
# on investment is approximately 7x, indicating that customer
# retention initiatives could provide significant financial value.

# These findings reinforce that preventing churn among
# previously high-value customers should be a strategic
# business priority.

## 7. Export & Day Summary

In [93]:
# Save for use in Day 5 segmentation and SQL
rfm_base.to_csv('rfm_base.csv', index=False)

print("RFM base table saved.")
print(f"\nFinal table shape: {rfm_base.shape}")
print(f"Columns: {list(rfm_base.columns)}")
print()

# Print a clean summary for your notebook
print("=" * 50)
print("SUMMARY")
print("=" * 50)
print(f"Customers scored:     {len(rfm_base):,}")
print(f"Recency range:        {rfm_base['Recency'].min()} to {rfm_base['Recency'].max()} days")
print(f"Frequency range:      {rfm_base['Frequency'].min()} to {rfm_base['Frequency'].max()} orders")
print(f"Monetary range:       £{rfm_base['Monetary'].min():.2f} to £{rfm_base['Monetary'].max():,.2f}")
print(f"Avg RFM Total score:  {rfm_base['RFM_Total'].mean():.1f}")
print(f"Highest scored:       {rfm_base['RFM_Total'].max()} (Champions territory)")
print(f"Lowest scored:        {rfm_base['RFM_Total'].min()} (Lost territory)")



RFM base table saved.

Final table shape: (5878, 14)
Columns: ['customer_id', 'last_purchase_date', 'first_purchase_date', 'Frequency', 'Monetary', 'avg_order_value', 'total_items', 'unique_products', 'Recency', 'tenure_days', 'R_Score', 'F_Score', 'M_Score', 'RFM_Total']

SUMMARY
Customers scored:     5,878
Recency range:        1 to 739 days
Frequency range:      1 to 398 orders
Monetary range:       £2.95 to £580,987.04
Avg RFM Total score:  9.0
Highest scored:       15 (Champions territory)
Lowest scored:        3 (Lost territory)
